# INTERVENE — write residual or logits

Same code as `scripts/intervene.py`, one **mode per cell**. Qwen weights stay frozen.

**Kernel:** `CXR local Qwen (faiss_gpu1)`

Run order:
1. **Setup**
2. **Load model** (skip if you already loaded it in another notebook in this kernel — this kernel is per notebook, so load once here)
3. Any mode cell below


## Setup


In [9]:
import json
import os
import sys
from pathlib import Path

SCRIPTS = Path("/home/udonsi-kalu/staging/cxr-mi-repeng-grounding/scripts")
CASES = Path("/home/udonsi-kalu/staging/cxr-mi-repeng-grounding/learning_lab/cases/oncology_m1.json")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import look
import intervene
import process
from _common import PROMPT_A, PROMPT_B, PROMPT_TEST, DEFAULT_MODEL, boot, encode, last_residual, device

cases = json.loads(CASES.read_text(encoding="utf-8"))
NOTE = cases["foundation_note"]
LAYER = 20
MAX_NEW = 24

print("Teaching note:", NOTE)
print("Imported look, intervene, process.")


Teaching note: Patient received FOLFOX. Disease progressed. FOLFOX was discontinued.
Imported look, intervene, process.


## Load model (run once)


In [10]:
model, tok = boot(DEFAULT_MODEL, layer=LAYER)


BACKEND  REUSE — not calling from_pretrained; inspecting live weights
  pid=4097531  class=Qwen2ForCausalLM
  name=Qwen/Qwen2.5-7B-Instruct
  torch_dtype=torch.float16  training=False
  blocks=28  hidden=3584  using layer 20
  tokenizer=Qwen2TokenizerFast  vocab=151643  pad=151643  eos=151645
  hf_device_map (accelerate placement):
    model.embed_tokens: 0
    model.layers.0: 0
    model.layers.1: 0
    model.layers.2: 0
    model.layers.3: 0
    model.layers.4: 0
    model.layers.5: 0
    model.layers.6: 0
    model.layers.7: 0
    model.layers.8: 0
    model.layers.9: 0
    model.layers.10: 0
    model.layers.11: 0
    model.layers.12: 0
    model.layers.13: 0
    model.layers.14: 0
    model.layers.15: cpu
    model.layers.16: cpu
    model.layers.17: cpu
    model.layers.18: cpu
    model.layers.19: cpu
    model.layers.20: cpu
    model.layers.21: cpu
    model.layers.22: cpu
    model.layers.23: cpu
    model.layers.24: cpu
    model.layers.25: cpu
    model.layers.26: cpu
    m

### `steer`


In [11]:
intervene.cmd_steer(model, tok, layer=LAYER, alpha=8.0, max_new=MAX_NEW)


steer L20 alpha=8.0
BASE    ' No. Based on the information provided, it seems that the patient has completed a course of oxaliplatin and '
STEER   ' No. Based on the information provided, it seems that the patient has completed a course of oxaliplatin and '


### `patch`


In [14]:
intervene.cmd_patch(model, tok, layer=LAYER, max_new=MAX_NEW)


patch L20: write A's last-token vector into B's forward
BASE B  ' No. The patient has not stopped FOLFOX; they are continuing with the treatment as planned for the current cycle.'
PATCH   ' No No No No No No No No No No No No No No No No No No No No No No No No'


### `mlp_zero`


In [12]:
intervene.cmd_zero(model, tok, layer=LAYER, max_new=MAX_NEW, site="mlp")


zero L20 mlp last-token write
BASE    ' No. Based on the information provided, it seems that the patient has completed a course of oxaliplatin and '
ZERO    ' No. The patient is on a maintenance regimen of FOLFOX, which typically refers to a combination of fluorourac'


### `attn_zero`


In [13]:
intervene.cmd_zero(model, tok, layer=LAYER, max_new=MAX_NEW, site="attn")


zero L20 attn last-token write
BASE    ' No. Based on the information provided, it seems that the patient has completed a course of oxaliplatin and '
ZERO    ' No. The information provided does not indicate that FOLFOX has stopped. It only mentions that an oxaliplatin'


### `resid_zero`


In [15]:
intervene.cmd_zero(model, tok, layer=LAYER, max_new=MAX_NEW, site="block")


zero L20 block last-token write
BASE    ' No. Based on the information provided, it seems that the patient has completed a course of oxaliplatin and '
ZERO    'ONUS.StartPosition.StartPosition.StartPosition.StartPosition.StartPosition.StartPosition.StartPosition.StartPosition dur dur dur dur dur dur <<<vantvantvantvantvantvantvantvant'


### `logit_bias`


In [17]:
intervene.cmd_logit_bias(model, tok, max_new=MAX_NEW)


BASE  ' No. Based on the information provided, it seems that the patient has completed a course of oxaliplatin and '
BIAS yes+4 no-4  ' Yes. Based on the information provided, the patient has completed a course of oxaliplatin and 5-FU'
